# 02 - Voter registration 2026

Owner: B

Goal: one row per municipality with registered voters by age group and gender, and the date of the snapshot.

Reads: `data/raw/registration/`
Writes: `INTERIM / "registration_2026.csv"`

Run the two setup cells first.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Check the downloads page first

If the 2026 entry on the IEC results downloads page already has registration figures, use it and skip the steps
below.

In [ ]:
# inv = inventory(RAW / "registration"); inv

## Find where the dashboard gets its data

In the browser: DevTools, Network, Fetch/XHR, then pick a municipality and copy the request as cURL. Turn it into
Python at curlconverter.com.

In [ ]:
import requests, time, json

# url = "..."            # from DevTools
# headers = {...}
# r = requests.get(url, headers=headers, timeout=30)
# r.json()

## Get every municipality

One request per municipality, with a one-second pause between requests. Save the raw answers before cleaning them.

In [ ]:
# rows = []
# for prov in provinces:
#     for muni in munis[prov]:
#         rows.append(fetch(prov, muni)); time.sleep(1)

## One row per municipality

Add `registered_youth` = ages 18 to 19 plus ages 20 to 29, and add the snapshot date.

In [ ]:
from datetime import date
# reg["registered_youth"] = reg["registered_18_19"] + reg["registered_20_29"]
# reg["snapshot_date"] = date.today().isoformat()

## Check

The national total should be close to 29.2 million (the figure on 21 September 2026). It goes up every day.

In [ ]:
# print(f'{reg["registered_total"].sum():,}')

## Save output

In [ ]:
# out.to_csv(INTERIM / "registration_2026.csv", index=False)
# print(out.shape); out.head()